# arange-fancy-index-cross-entropy — worked example 3: NumPy: gather predicted-class probability and check correctness

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `arange-fancy-index-cross-entropy`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The same `arange` fancy-index pattern works in NumPy: `arr[np.arange(B), idx]` reads one column per row. Here we use it to pull the model's probability assigned to the true class, and separately compare argmax predictions to targets.

## Worked solution

**Goal.** Given a `(B, C)` probability matrix `probs` (rows sum to 1) and integer `labels` of shape `(B,)`, return the `(B,)` vector of probabilities the model assigned to the correct class, using NumPy fancy indexing.

**Step 1 — row index.** `np.arange(B)` is `[0, 1, ..., B-1]`. In NumPy, indexing a 2-D array with two equal-length 1-D integer arrays pairs them element-wise: result entry `i` is `probs[i, labels[i]]`.

**Step 2 — gather.** `probs[np.arange(B), labels]` yields the `(B,)` vector of true-class probabilities. This avoids any Python loop.

**Step 3 — sanity exercise.** We also compute the argmax prediction per row with `probs.argmax(axis=1)` and compare to `labels` to get an accuracy scalar — a common downstream use of the same per-row reasoning.

**Why this matters.** The gather is the NumPy mirror of the PyTorch `t.arange(B)` idiom; the mechanics (positional pairing of two index arrays) are identical across the two libraries.

In [ ]:
def true_class_probs(probs, labels):
    B = probs.shape[0]
    return probs[np.arange(B), labels]            # (B,)

np.random.seed(0)
raw = np.random.rand(5, 4)
probs = raw / raw.sum(axis=1, keepdims=True)      # rows sum to 1
labels = np.array([1, 3, 0, 2, 1])
gathered = true_class_probs(probs, labels)
acc = (probs.argmax(axis=1) == labels).mean()
print(gathered.shape)
print(np.round(gathered, 4))
print('accuracy', acc)